# DINO-RAD (RAD-DINO) Classification for Kaggle (GPU)

This notebook is optimized for running on **Kaggle** using a **GPU (P100 or T4)**.
It uses Microsoft's **RAD-DINO** model for medical image classification.

In [ ]:
# --- 1. Dependencies and Setup ---
import os
import sys
import torch

# Ensure compatibility with multiple transformers versions
!pip install -U transformers datasets evaluate accelerate torch torchvision

import transformers
from datasets import load_dataset
from transformers import (
    AutoImageProcessor, 
    AutoModelForImageClassification, 
    TrainingArguments, 
    Trainer
)
import evaluate
import numpy as np

print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

model_id = "microsoft/rad-dino"

# FLEXIBLE DATASET PATH FOR LOCAL OR KAGGLE
if os.path.exists("/kaggle/input"):
    # CHANGE THIS to your specific Kaggle dataset path
    dataset_root = "/kaggle/input/fracture-dataset/balanced_augmented_dataset"
else:
    dataset_root = r"c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\balanced_augmented_dataset"

print(f"Using dataset root: {dataset_root}")

In [ ]:
# --- 2. Load Dataset ---
if not os.path.exists(dataset_root):
    raise FileNotFoundError(f"Dataset not found at {dataset_root}. Please check the path.")

dataset = load_dataset(
    "imagefolder", 
    data_files={
        "train": os.path.join(dataset_root, "train/**"),
        "validation": os.path.join(dataset_root, "val/**"),
        "test": os.path.join(dataset_root, "test/**")
    }
)

labels = dataset["train"].features["label"].names
label2id, id2label = {label: i for i, label in enumerate(labels)}, {i: label for i, label in enumerate(labels)}

print(f"Classes: {labels}")

In [ ]:
# --- 3. Preprocessing ---
image_processor = AutoImageProcessor.from_pretrained(model_id)

def transform(example_batch):
    """Apply the DINO-RAD required preprocessing to images."""
    inputs = image_processor([x.convert("RGB") for x in example_batch["image"]], return_tensors="pt")
    inputs["labels"] = example_batch["label"]
    return inputs

prepared_ds = dataset.with_transform(transform)

In [ ]:
# --- 4. Load Model ---
model = AutoModelForImageClassification.from_pretrained(
    model_id, 
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

In [ ]:
# --- 5. Metrics ---
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
# --- 6. Training Arguments ---
output_dir = "./outputs/dinorad_lib"

training_args_kwargs = {
    "output_dir": output_dir,
    "remove_unused_columns": False,
    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "per_device_eval_batch_size": 16,
    "num_train_epochs": 3,
    "logging_steps": 10,
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy",
    "push_to_hub": False,
    "report_to": "none",
    "fp16": torch.cuda.is_available(), # Enable mixed precision if GPU is available
}

# Compatibility fix for Transformers 5.x
if hasattr(TrainingArguments, "use_cpu"):
    training_args_kwargs["use_cpu"] = not torch.cuda.is_available()
else:
    training_args_kwargs["no_cuda"] = not torch.cuda.is_available()

training_args = TrainingArguments(**training_args_kwargs)

# --- 7. Trainer ---
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": prepared_ds["train"],
    "eval_dataset": prepared_ds["validation"],
    "compute_metrics": compute_metrics,
}

# Compatibility fix for Transformers 5.x (tokenizer -> processing_class)
import inspect
trainer_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = image_processor
else:
    trainer_kwargs["tokenizer"] = image_processor

trainer = Trainer(**trainer_kwargs)

# --- 8. Train! ---
trainer.train()

# Final Test Evaluation
test_metrics = trainer.evaluate(prepared_ds["test"])
print(f"Final Test Accuracy: {test_metrics['eval_accuracy']:.2%}")

# Save model
trainer.save_model(os.path.join(output_dir, "final_model"))